# Starter Template

**Your bot, from nothing to a submission.**

This notebook is the whole pipeline. You will come back to it across both
workshops, filling in one part at a time:

| Part | What you build | Covered in |
|---|---|---|
| 1 | Scrape both websites | Workshop 1, block 4 |
| 2 | Chunk, embed and index | Workshop 1, block 4 |
| 3 | Describe and index images | Workshop 2, blocks 2 and 3 |
| 4 | Answer a question | Workshop 1, blocks 2 and 3 |
| 5 | Test against the dev set | Workshop 1, block 5 |
| 6 | Export `trivia.py` for submission | before 21 October |

Everything marked `TODO` is yours. Everything else works already.

---

## The one contract that matters

Codabench runs your bot as a command line program:

```
python trivia.py "What faculty does the Innovation Wing belong to?"
```

It passes one question as an argument and reads the answer from stdout.
Part 6 writes that file for you from the functions you define here, so
you never have to maintain the same code twice.

Two functions must exist:

```python
rag_answer(question: str) -> str            # one in, one out
rag_answer_batch(questions: list) -> list   # same length, same order
```

The names, the inputs and the outputs are fixed. Everything inside them
is yours.

> **Before you start:** nothing. This notebook is where your bot begins.
>
> **When you finish Part 2:** `data/chroma` exists and labs 3, 4 and 5 can read it.
>
> **When you finish Part 6:** `trivia.py` is ready to submit.

---
## Setup

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops.

**Do not commit this notebook with a key visible in its output.** The
exported `trivia.py` reads the key from an environment variable instead,
so your submission never contains it.

In [ ]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

In [ ]:
# ---- given helpers. You should not need to change these. ----------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def describe_image_bytes(raw, mime, prompt):
    """Describe one image given its raw bytes."""
    import base64
    b64 = base64.b64encode(raw).decode()
    r = vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P
    if reset and _P(path).exists():
        shutil.rmtree(path)          # cleaner than deleting the collection
    try:
        c = chromadb.PersistentClient(path=path)
        return c.get_or_create_collection(name)
    except KeyError as e:
        raise RuntimeError(
            f"chromadb cannot read the index at {path} ({e}). It was built by a "
            f"different chromadb version. Delete that folder and rebuild, or "
            f"install the pinned version from requirements.txt."
        ) from None


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")

print("helpers loaded")

---
# Part 1: Scrape

**Workshop 1, block 4.**

Two functions to write. `crawl` finds every page; `extract` pulls the
readable text and the images off one of them.

Check for a sitemap before writing a crawler. If `/sitemap.xml` exists it
lists every page and you can skip the crawl entirely.

**The trap:** `soup.get_text()` on the whole document returns the
navigation menu, header and footer from every page. Those near-identical
fragments become chunks that look moderately similar to every query, and
they will push real results out of your top five. Open the site in a
browser, right click the content, choose Inspect, and find the element
that wraps it.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

SITES = [
    "https://innowings.engg.hku.hk/innowing1/",
    "https://innoacademy.engg.hku.hk/",
]


def crawl(start_url, max_pages=500):
    """Return every page URL on the same site as start_url."""
    # TODO check for a sitemap first: requests.get(start_url + "sitemap.xml")
    # TODO follow internal links only, and keep a visited set to avoid loops
    seen, queue, out = set(), [start_url], []
    domain = urlparse(start_url).netloc

    while queue and len(out) < max_pages:
        url = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)
        try:
            html = requests.get(url, timeout=20).text
        except Exception:
            continue
        out.append(url)

        # TODO find the links on this page and add the internal ones to queue
        # for a in BeautifulSoup(html, "html.parser").select("a[href]"):
        #     link = urljoin(url, a["href"]).split("#")[0]
        #     if urlparse(link).netloc == domain and link not in seen:
        #         queue.append(link)

    return out


def extract(html, url):
    """Return {"url", "title", "text", "images": [...]} for one page."""
    soup = BeautifulSoup(html, "html.parser")

    # TODO replace this with the element that actually holds the content.
    # soup.select_one("main") or soup.select_one("#content") or similar.
    body = soup                                  # <-- currently the whole page

    images = []
    for img in soup.select("img"):
        src = img.get("src")
        if not src:
            continue
        fig = img.find_parent("figure")
        images.append({
            "src":     urljoin(url, src),        # relative -> absolute
            "alt":     img.get("alt", ""),
            "caption": (fig.find("figcaption").get_text(strip=True)
                        if fig and fig.find("figcaption") else ""),
            "page":    url,
        })

    return {
        "url":    url,
        "title":  soup.title.get_text(strip=True) if soup.title else "",
        "text":   body.get_text(" ", strip=True),
        "images": images,
    }

In [ ]:
# Run the scrape and save it. This takes a few minutes.
pages = []
for site in SITES:
    for url in crawl(site):
        try:
            pages.append(extract(requests.get(url, timeout=20).text, url))
        except Exception as exc:
            print("skipped", url, exc)

Path("data").mkdir(exist_ok=True)
Path("data/pages.json").write_text(json.dumps(pages, indent=1))

images = [im for p in pages for im in p["images"]]
Path("data/images.json").write_text(json.dumps(images, indent=1))

print(f"{len(pages)} pages, {len(images)} images")
print("\nCheck one before moving on:")
print(pages[0]["text"][:400] if pages else "(nothing scraped)")

### Before you move on

Read that sample text. If it starts with your navigation menu, fix
`extract` now. Every chunk you build from here inherits the problem, and
finding it later means rebuilding the whole index.

---
# Part 2: Index

**Workshop 1, block 4.**

Chunk the text, embed it, write it to the store.

**Store the metadata now.** Level 4 questions need to filter by year and
page type, and adding a field later means rebuilding everything. It costs
nothing to store fields you cannot yet see a use for.

In [ ]:
CONFIG = {
    "chunk_size": 800,     # try 400 to 1200
    "overlap":    100,     # try 0 to 20% of chunk size
    "k":          5,       # try 3 to 10
}


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap.

    Overlap exists because a fact split across a boundary is lost: a date
    at char 998 and its event name at 1002 land in different chunks and
    neither answers the question.
    """
    # TODO try a better strategy once this works. Split on headings first,
    # TODO fall back to characters last, and prepend the section heading.
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


def build_index(pages, config, reset=True):
    texts, metas = [], []
    for page in pages:
        for i, piece in enumerate(chunk(page["text"],
                                        config["chunk_size"], config["overlap"])):
            texts.append(piece)
            metas.append({
                "url":      page["url"],
                "title":    page["title"],
                "position": i,
                "kind":     "text",
                # TODO add "year" and "page_type" here. Level 4 needs them.
            })
    store = get_store(reset=reset)
    add_to_store(store, texts, metas)
    print(f"indexed {len(texts)} chunks")
    return store


pages = json.loads(Path("data/pages.json").read_text())
store = build_index(pages, CONFIG)

---
# Part 3: Images

**Workshop 2, blocks 2 and 3.**

The answer to a Level 3 question lives inside a JPEG. No amount of
chunking or reranking reaches it, because it was never turned into text.

Describe each image **at ingestion**, store the description as an
ordinary chunk beside your scraped text, and retrieval works unchanged.
Never describe an image while answering a question: ingestion time is
unlimited, runtime is 30 seconds.

The prompt below is deliberately empty. Writing it is the Workshop 2
block 3 exercise, and it sets the ceiling on your Level 3 and Level 5
scores.

In [ ]:
DESCRIPTION_PROMPT = """
TODO [W2 b3] Write this.

A caption written for a human ("a room with modern furniture and
students working") cannot answer "how many tables are in Makerspace A"
or "what three words are on the wall behind the brainstorming area".

What would this prompt have to ask for so that both are answerable?
"""


def describe_all(images, cache_path="data/descriptions.json"):
    """Describe every image once, cache the result, never regenerate.

    Ingestion is free, but not if you redo it every time you change a
    line downstream. Key the cache by image URL.
    """
    cache = {}
    p = Path(cache_path)
    if p.exists():
        cache = json.loads(p.read_text())

    for im in images:
        src = im["src"]
        if src in cache:
            continue
        try:
            r = requests.get(src, timeout=30)
            cache[src] = describe_image_bytes(
                r.content,
                r.headers.get("content-type", "image/jpeg").split(";")[0],
                DESCRIPTION_PROMPT,
            )
        except Exception as exc:
            print("failed:", src[:70], exc)

    p.write_text(json.dumps(cache, indent=1))
    print(f"{len(cache)} descriptions cached")
    return cache


def index_descriptions(descriptions, images, store):
    """Add descriptions to the same store as the scraped text."""
    by_src = {im["src"]: im for im in images}
    texts, metas = [], []
    for src, text in descriptions.items():
        im = by_src.get(src, {})
        # Alt text and captions are already text and cost nothing to index.
        full = " ".join(filter(None, [im.get("alt"), im.get("caption"), text]))
        texts.append(full)
        metas.append({"url": im.get("page", src), "image": src, "kind": "image"})
    add_to_store(store, texts, metas,
                 ids=[f"img_{i}" for i in range(len(texts))])
    print(f"indexed {len(texts)} image descriptions")


images       = json.loads(Path("data/images.json").read_text())
descriptions = describe_all(images)
index_descriptions(descriptions, images, store)

---
# Part 4: Answer

**Workshop 1, blocks 2 and 3.**

Three functions. `retrieve` finds the chunks, `rag_answer` answers one
question, `rag_answer_batch` answers several.

`retrieve` is separate on purpose. Keeping it lets you check whether the
answer was even fetched, which is the only way to tell a retrieval
failure from a prompt failure.

### On the prompt

| Part | Why |
|---|---|
| Grounding | Models otherwise fall back on training data |
| Format | Judges mark accuracy and completeness. Extra wrong detail turns 1 into 0.5 |
| Fallback | A refusal and a wrong answer both score zero, so always guess |

In [ ]:
SYSTEM_PROMPT = """You answer questions about the Tam Wing Fan Innovation Wing.

Answer only from the context below. Where the context disagrees with what
you think you know, the context is correct.

Reply with the answer only. No explanation, no preamble. If the question
asks how many, reply with a number.

If the context does not contain the answer, give your best guess anyway.
Never reply that you do not know."""


def retrieve(question, k=5, where=None):
    """Return the k chunks most relevant to the question.

    Each dict has "text", "metadata" and "distance". Chroma returns
    squared L2, so lower is closer.
    """
    store = get_store()
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def rag_answer(question):
    """One question in, one answer out. 30 second budget."""
    chunks = retrieve(question, k=CONFIG["k"])

    # TODO once this works: decompose compound questions, retrieve wide
    # TODO then filter, or filter by metadata before searching.

    context = "\n\n".join(
        f"[{c['metadata'].get('url', '?')}]\n{c['text']}" for c in chunks)

    reply = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

    # TODO check the format of what came back: if you asked for a
    # TODO number, make sure you got one and strip any stray prose.
    return reply.strip()


def rag_answer_batch(questions):
    """Many questions in, the same number of answers out, in order.

    A loop is correct and is all most teams need. Replace it if you can
    share work: one embedding call for every query rather than one per
    query, the store opened once, sub-queries running concurrently.
    """
    return [rag_answer(q) for q in questions]


print(rag_answer("What faculty does the Tam Wing Fan Innovation Wing belong to?"))

---
# Part 5: Test

**Workshop 1, block 5.**

Run the dev set, then **read the answers**. Mark each one yourself: 1
correct, 0.5 partly, 0 wrong. Fifteen questions takes about three
minutes, and reading them is how you notice the bot answered a slightly
different question, hedged, or returned three paragraphs.

`retrieved` is checked automatically. It is the ceiling on your score: if
the answer was never fetched, no prompt can recover it.

In [ ]:
def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def run_dev_set(path="dev_set.json"):
    dev, rows, times = json.loads(Path(path).read_text()), [], []
    for item in dev:
        t0 = time.time()
        chunks = retrieve(item["question"], k=CONFIG["k"])
        given  = rag_answer(item["question"])
        secs   = time.time() - t0
        times.append(secs)
        rows.append({**item, "given": given, "chunks": chunks, "seconds": secs})

    got = sum(answer_present(r["answer"], r["chunks"]) for r in rows)
    print(f"retrieved {got}/{len(rows)}   "
          f"median {statistics.median(times):.1f}s   slowest {max(times):.1f}s")
    if max(times) > 30:
        print("WARNING: at least one question is over the 30 second limit.")

    for r in rows:
        flag = "yes" if answer_present(r["answer"], r["chunks"]) else "NO"
        print(f"\nLv{r['level']}  {r['question']}")
        print(f"   expected:  {r['answer']}")
        print(f"   got:       {r['given'] or '(empty)'}")
        print(f"   retrieved: {flag}    {r['seconds']:.1f}s")
    return rows


rows = run_dev_set()

---
# Part 6: Export your submission

Writes a standalone `trivia.py` containing the functions you defined
above, ready for Codabench.

Two things the export handles for you:

- **Your key is not written into the file.** The exported script reads
  `AZURE_OPENAI_KEY` from the environment instead.
- **The command line contract is added automatically**, including the
  guard that prints an empty answer rather than crashing, so one bad
  question cannot zero a whole run.

Test the exported file from a terminal before submitting. A bot that
works in a notebook and fails from the command line scores zero.

In [ ]:
import inspect

EXPORT = [
    chat, embed, get_store,
    chunk, retrieve, rag_answer, rag_answer_batch,
]

HEADER = f'''"""Chatbot Challenge 2026 submission. Exported from the starter notebook."""
import os, sys, time
from openai import AzureOpenAI

API_VERSION       = "{API_VERSION}"
CHAT_DEPLOYMENT   = "{CHAT_DEPLOYMENT}"
VISION_DEPLOYMENT = "{VISION_DEPLOYMENT}"
EMBED_DEPLOYMENT  = "{EMBED_DEPLOYMENT}"
CHAT_URL          = "{CHAT_URL}"
VISION_URL        = "{VISION_URL}"
EMBED_URL         = "{EMBED_URL}"
CONFIG            = {CONFIG!r}
TIME_LIMIT        = 30

# The gateway takes the full path as the endpoint, so one client per
# deployment rather than one per gateway.
_key = os.environ["AZURE_OPENAI_KEY"].strip()
_c = lambda u: AzureOpenAI(azure_endpoint=u, api_key=_key, api_version=API_VERSION)
chat_client, vision_client, embed_client = _c(CHAT_URL), _c(VISION_URL), _c(EMBED_URL)

SYSTEM_PROMPT = """{SYSTEM_PROMPT}"""
'''

FOOTER = '''

def main():
    questions = sys.argv[1:]
    if not questions:
        print(\'usage: python trivia.py "question" ["question" ...]\', file=sys.stderr)
        sys.exit(1)

    started = time.time()
    try:
        answers = ([rag_answer(questions[0])] if len(questions) == 1
                   else list(rag_answer_batch(questions)))
    except Exception as exc:
        print(f"[error] {type(exc).__name__}: {exc}", file=sys.stderr)
        answers = [""] * len(questions)

    if len(answers) != len(questions):
        answers = (answers + [""] * len(questions))[:len(questions)]

    per_q = (time.time() - started) / len(questions)
    if per_q > TIME_LIMIT:
        print(f"[warning] {per_q:.1f}s per question", file=sys.stderr)

    for a in answers:
        print("" if a is None else str(a))


if __name__ == "__main__":
    main()
'''

parts = []
for f in EXPORT:
    try:
        parts.append(inspect.getsource(f))
    except OSError:
        raise RuntimeError(
            f"Could not read the source of {f.__name__}(). This happens when "
            f"the kernel was restarted and that cell was not re-run. "
            f"Use Run > Run All Above, then run this cell again."
        )

body = "\n\n".join(parts)
Path("trivia.py").write_text(HEADER + "\n\n" + body + FOOTER)

print("wrote trivia.py")
print("\nTest it from a terminal, not from here:")
print('  export AZURE_OPENAI_KEY="your-key"')
print('  python trivia.py "What faculty does the Innovation Wing belong to?"')

---
## Before you submit

- [ ] `python trivia.py "question"` works from a terminal, not just here
- [ ] It prints one line, and nothing else, to stdout
- [ ] Every question comes back under 30 seconds
- [ ] `data/chroma` is committed. It is generated output, so the instinct
      is to gitignore it, but without it your bot has nothing to retrieve
      from on the grading machine
- [ ] `requirements.txt` is pinned with `pip freeze`, and you have tested
      a clean install in a fresh virtual environment
- [ ] No API key anywhere in the repository, including in notebook output

The competition runs on Digital Learning Studio machines with a 30 minute
install window. A dependency that resolves differently on the day costs
you the round.